# 09 — Hybrid Anomaly Detection & Ablation Study
## Team T033 — Project 26: Card Fraud Triage — Phase 2
**Author:** Prachee Dhar (Task P2_04)

This notebook demonstrates the complete H2 (Score Fusion) and H3 (DAGMM-lite) hybrid pipeline,
visualizes ablation results, and presents the master evaluation table comparing all Phase 1 and Phase 2 models.

---
**Ownership reminder:** This notebook calls `src/hybrid_fusion.py`, `src/hybrid_dagmm_lite.py`, and `src/ablation.py`.
It does NOT modify Vani's AE (`src/autoencoder.py`) or Apoorva's Latent-GMM (`src/latent_gmm.py`).

In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')

# Add repo root to path
REPO = Path('.').resolve()
while not (REPO / 'src').exists() and REPO != REPO.parent:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

print(f'Repo root: {REPO}')

In [ ]:
from src.config import CFG, get_repo_root
from src.data import get_features
from src.utils import load_scores
from src.metrics import pr_auc, recall_at_fpr
from src.evaluation import bootstrap_ci

# Load data splits
splits = get_features('base')
y_val = splits.y_val
y_test = splits.y_test
print(f'Val: {len(y_val)} samples, {y_val.sum()} frauds')
print(f'Test: {len(y_test)} samples, {y_test.sum()} frauds')

## 1. Load All Score Files

In [ ]:
score_names = [
    ('PCA Anomaly', 'pca'),
    ('LOF', 'lof'),
    ('GMM (ambient)', 'gmm'),
    ('OC-SVM', 'ocsvm'),
    ('Isolation Forest', 'iforest'),
    ('AE', 'ae'),
    ('DAE', 'ae_dae'),
    ('H1: Latent GMM', 'hybrid_latentgmm'),
    ('H2: Score Fusion', 'hybrid_fusion'),
    ('H3: DAGMM-lite', 'hybrid_dagmmlite'),
]

scores = {}
for label, name in score_names:
    try:
        s_v = load_scores(name, 'val')
        s_t = load_scores(name, 'test')
        scores[label] = (s_v, s_t)
        val_pr = pr_auc(y_val, s_v)
        test_pr = pr_auc(y_test, s_t)
        print(f'{label:25s}: Val PR-AUC={val_pr:.4f} | Test PR-AUC={test_pr:.4f}')
    except FileNotFoundError:
        print(f'{label:25s}: MISSING')

## 2. H2 Fusion Weight Sensitivity

In [ ]:
df_weights = pd.read_csv(REPO / 'results/tables/hybrid_fusion_weights.csv', index_col=0)
df_weights.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for norm_name, grp in df_weights.groupby('norm_method'):
    grp = grp.sort_values('alpha')
    ax.plot(grp['alpha'], grp['val_pr_auc'], 'o--', lw=2, label=f'norm={norm_name}')

ax.axvline(1.0, color='crimson', ls=':', lw=2, label='Selected alpha=1.0')
ax.set_xlabel('Alpha (DAE weight; GMM weight = 1-alpha)', fontsize=12)
ax.set_ylabel('Val PR-AUC', fontsize=12)
ax.set_title('H2 Score Fusion: Weight Sensitivity (Val PR-AUC vs Alpha)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(REPO / 'results/figures/hybrid_fusion_sensitivity_nb.png', dpi=150, bbox_inches='tight')
plt.show()
print('Finding: alpha=1.0 (pure DAE) is optimal on val because GMM val PR-AUC is only 0.1038')

## 3. H2 Score Correlation Analysis

In [ ]:
df_corr = pd.read_csv(REPO / 'results/tables/hybrid_fusion_correlation.csv', index_col=0)
print('Score correlation (DAE vs GMM):')
df_corr

## 4. H3 DAGMM-lite Grid Search

In [ ]:
df_grid = pd.read_csv(REPO / 'results/tables/hybrid_dagmmlite_grid.csv', index_col=0)
print(f'Grid search: {len(df_grid)} configurations')
print(f'Best config (highest val PR-AUC):')
print(df_grid.loc[df_grid['val_pr_auc'].idxmax()])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, cov in zip(axes, ['diag', 'tied']):
    sub = df_grid[df_grid['covariance_type'] == cov].sort_values('k')
    for reg, grp in sub.groupby('reg_covar'):
        ax.plot(grp['k'], grp['val_pr_auc'], 'o--', label=f'reg={reg:.2e}')
    ax.set_xlabel('K (components)', fontsize=11)
    ax.set_ylabel('Val PR-AUC', fontsize=11)
    ax.set_title(f'H3 DAGMM-lite Grid: cov={cov}', fontsize=12)
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.5)

plt.suptitle('H3 DAGMM-lite: Hyperparameter Search (Val PR-AUC)', fontsize=13)
plt.tight_layout()
plt.savefig(REPO / 'results/figures/hybrid_dagmmlite_grid_nb.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Fusion Method Comparison

In [ ]:
df_methods = pd.read_csv(REPO / 'results/tables/hybrid_fusion_methods.csv', index_col=0)
print('H2 Fusion Method Comparison:')
df_methods[['fusion_method', 'val_pr_auc', 'test_pr_auc', 'val_recall_fpr05', 'test_recall_fpr05']]

## 6. Alert Overlap / Venn Analysis

In [ ]:
df_overlap = pd.read_csv(REPO / 'results/tables/hybrid_overlap.csv', index_col=0)
print('Top-100 Alert Overlap (Test Set):')
df_overlap[df_overlap['capacity_k'] == 100]

In [ ]:
df_venn = pd.read_csv(REPO / 'results/tables/hybrid_overlap_venn_breakdown.csv', index_col=0)
print('3-way Venn Breakdown (DAE / GMM / H2 Fusion) @ Top-100:')
df_venn.T

## 7. Master Comparison Table

In [ ]:
df_master = pd.read_csv(REPO / 'results/tables/eval_master_phase2.csv', index_col=0)
cols = ['model', 'val_pr_auc', 'test_pr_auc', 'test_pr_auc_ci_lo', 'test_pr_auc_ci_hi',
        'test_recall_fpr01', 'test_recall_fpr05', 'test_prec_100']
df_display = df_master[cols].copy()
df_display['CI_95'] = df_display.apply(lambda r: f"[{r['test_pr_auc_ci_lo']:.3f}, {r['test_pr_auc_ci_hi']:.3f}]", axis=1)
df_display = df_display.drop(columns=['test_pr_auc_ci_lo', 'test_pr_auc_ci_hi'])
df_display = df_display.sort_values('test_pr_auc', ascending=False).reset_index(drop=True)
print('Master Evaluation (sorted by Test PR-AUC):')
df_display

In [ ]:
# Horizontal bar chart
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#2b5c8f' if 'P2' in row['model'] else '#8c8c8c' for _, row in df_master.iterrows()]
ax.barh(df_master['model'], df_master['test_pr_auc'], color=colors, edgecolor='black', alpha=0.85)
xerr_lo = df_master['test_pr_auc'] - df_master['test_pr_auc_ci_lo']
xerr_hi = df_master['test_pr_auc_ci_hi'] - df_master['test_pr_auc']
ax.errorbar(df_master['test_pr_auc'], df_master['model'], 
            xerr=[xerr_lo.values, xerr_hi.values], fmt='none', ecolor='black', capsize=4, lw=1.5)
ax.set_xlabel('Test PR-AUC (95% bootstrap CI)', fontsize=12)
ax.set_title('Phase 1 & 2 Master Evaluation: Test PR-AUC\n(Blue = Phase 2, Gray = Phase 1)', fontsize=12)
ax.grid(True, axis='x', linestyle='--', alpha=0.5)
for i, row in df_master.iterrows():
    ax.annotate(f"{row['test_pr_auc']:.4f}", xy=(row['test_pr_auc'] + 0.01, i), va='center', fontsize=9)
plt.tight_layout()
plt.savefig(REPO / 'results/figures/master_comparison_nb.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Ablation Summary

In [ ]:
df_abl_summary = pd.read_csv(REPO / 'results/tables/ablation_summary.csv', index_col=0)
print('Ablation Summary:')
df_abl_summary[['factor', 'best_config', 'val_pr_auc_best', 'delta_val', 'finding']]

In [ ]:
# A1: AE vs DAE bar chart
df_a1 = pd.read_csv(REPO / 'results/tables/ablation_ae_vs_dae.csv', index_col=0)
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(df_a1))
ax.bar(x - 0.18, df_a1['val_pr_auc'], 0.35, label='Val PR-AUC', color='steelblue', alpha=0.85, edgecolor='black')
ax.bar(x + 0.18, df_a1['test_pr_auc'], 0.35, label='Test PR-AUC', color='darkorange', alpha=0.85, edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(['AE (no noise)', 'DAE (sigma=0.2)'])
ax.set_ylabel('PR-AUC')
ax.set_title('A1: AE vs DAE — Denoising Regularization Effect')
ax.legend()
ax.grid(True, axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(REPO / 'results/figures/ablation_ae_vs_dae_nb.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"DAE vs AE delta val: +{df_a1['val_pr_auc'].iloc[1] - df_a1['val_pr_auc'].iloc[0]:.4f}")
print(f"DAE vs AE delta test: +{df_a1['test_pr_auc'].iloc[1] - df_a1['test_pr_auc'].iloc[0]:.4f}")

## 9. Paired Bootstrap Significance Tests

In [ ]:
df_paired = pd.read_csv(REPO / 'results/tables/hybrid_paired_bootstrap.csv', index_col=0)
print('Paired Bootstrap: Hybrid vs Standalone AE and GMM (Test PR-AUC):')
df_paired

## 10. Summary of Key Findings

| Finding | Value |
|---------|-------|
| Best overall model | PCA Anomaly Score (Test PR-AUC = 0.7425) |
| Best Phase 2 model (PR-AUC) | DAE / H2 (Test PR-AUC = 0.4818) |
| Best Phase 2 model (Recall@0.5% FPR) | H3 DAGMM-lite (Recall = 0.800) |
| H2 selected alpha | 1.0 (GMM val PR-AUC too low to contribute) |
| H3 selected config | K=16, diag covariance, reg=0.1 |
| A1: Denoising effect | +29.2% val PR-AUC (AE→DAE) |
| A6: Seed std | 0.148 (val), 0.078 (test) — sparse fraud labels cause noise |
| Leakage controls | All params fitted on train legit; val used for selection only |